In [97]:
import json
import os
import pandas as pd
import re
import shutil
from collections import defaultdict
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

In [51]:
p2_label_path = "chia_label/p2"
ready_path = "model_output/Llama-3-8B-Instruct_3_shot/ready"
failed_model_path = "model_output/Llama-3-8B-Instruct_3_shot/failed_inner"
p2_model_formatted_path = "model_output/Llama-3-8B-Instruct_3_shot/formatted"

In [52]:
def extract_nct_number(filename):
    parts = filename.split('_')
    nct_number = None
    file_type = None
    for part in parts:
        if part.startswith("NCT"):
            nct_number = part
        if part in ["inc", "exc"]:
            file_type = part
    return nct_number+"_"+file_type

def read_json(file_path):
    with open(file_path, 'r', encoding='utf-8') as file:
        return json.load(file)



def extract_logical_structure(data):
    structure = defaultdict(int)

    def traverse(node, depth=0):
        nonlocal structure
        structure["depth"] = max(structure["depth"], depth)

        if isinstance(node, dict):
            for key in node:
                if key in ["AND", "OR", "NOT"]:
                    structure[key] += 1
                traverse(node[key], depth + 1)
        elif isinstance(node, list):
            for item in node:
                traverse(item, depth + 1)

    traverse(data)
    return dict(structure)

In [73]:
label_files = {extract_nct_number(f): os.path.join(p2_label_path, f) for f in os.listdir(p2_label_path) if f.endswith('.json')}
model_files = {extract_nct_number(f): os.path.join(p2_model_formatted_path, f) for f in os.listdir(p2_model_formatted_path) if f.endswith('.json')}

common_ncts = set(label_files.keys()).intersection(model_files.keys())

In [88]:
labels = []
predictions = []
success_data = []

In [89]:

for nct in common_ncts:# ["NCT00050349_exc"]: #
    try:
        label_data = read_json(label_files[nct])
        model_data = read_json(model_files[nct])

        label_structure = extract_logical_structure(label_data)
        model_structure = extract_logical_structure(model_data)
        print(label_structure)
        success_data.append({
            'NCT': nct,
            'label_AND': label_structure.get('AND', 0),
            'label_OR': label_structure.get('OR', 0),
            'label_NOT': label_structure.get('NOT', 0),
            'label_DEPTH': label_structure.get('depth', 0),
            'model_AND': model_structure.get('AND', 0),
            'model_OR': model_structure.get('OR', 0),
            'model_NOT': model_structure.get('NOT', 0),
            'model_DEPTH': model_structure.get('depth', 0)
        })

        labels.append(label_structure)
        predictions.append(model_structure)
        #shutil.copy(model_files[nct], os.path.join(ready_path, os.path.basename(model_files[nct])))
    except Exception as e:
        print(f"Error processing NCT {nct}: {e}")
        #shutil.copy(model_files[nct], os.path.join(failed_model_path, os.path.basename(model_files[nct])))

{'depth': 39, 'AND': 22, 'OR': 7, 'NOT': 1}
Error processing NCT NCT02015923_exc: Expecting ',' delimiter: line 73 column 57 (char 2024)
{'depth': 7, 'AND': 3, 'NOT': 1}
{'depth': 7, 'AND': 2, 'OR': 1}
{'depth': 11, 'AND': 8, 'OR': 1}
{'depth': 7, 'AND': 3}
{'depth': 7, 'AND': 5}
{'depth': 9, 'AND': 4}
Error processing NCT NCT02567214_inc: Expecting property name enclosed in double quotes: line 31 column 17 (char 837)
{'depth': 17, 'AND': 6, 'OR': 3}
{'depth': 11, 'AND': 5}
{'depth': 11, 'AND': 5, 'OR': 1}
{'depth': 17, 'AND': 9}
{'depth': 11, 'AND': 5, 'OR': 1}
{'depth': 11, 'AND': 6}
{'depth': 17, 'AND': 5, 'OR': 4}
{'depth': 19, 'AND': 3, 'NOT': 1, 'OR': 6}
{'depth': 7, 'AND': 3}
Error processing NCT NCT01994382_inc: Extra data: line 117 column 1 (char 5136)
{'depth': 7, 'AND': 3}
{'depth': 5, 'AND': 1, 'OR': 1}
{'depth': 53, 'AND': 27, 'OR': 20, 'NOT': 2}
{'depth': 11, 'AND': 6, 'OR': 3}
{'depth': 11, 'AND': 5}
{'depth': 7, 'AND': 2, 'OR': 1}
{'depth': 29, 'AND': 7, 'NOT': 1, 'OR':

In [98]:
df_success = pd.DataFrame(success_data).set_index('NCT')

In [102]:
true_values = df_success[['label_AND', 'label_OR', 'label_NOT', 'label_DEPTH']].values
predicted_values = df_success[['model_AND', 'model_OR', 'model_NOT', 'model_DEPTH']].values

metrics = {}
for i, metric in enumerate(['AND', 'OR', 'NOT', 'DEPTH']):
    y_true = true_values[:, i]
    y_pred = predicted_values[:, i]

    metrics[metric] = {
        'accuracy': round(accuracy_score(y_true, y_pred), 3),
        'precision': round(precision_score(y_true, y_pred, average='macro', zero_division=0), 3),
        'recall': round(recall_score(y_true, y_pred, average='macro', zero_division=0), 3),
        'f1_score': round(f1_score(y_true, y_pred, average='macro', zero_division=0), 3),
        'confusion_matrix': confusion_matrix(y_true, y_pred)
    }

# Ausgabe der Metriken
for metric, values in metrics.items():
    print(f"Metrics for {metric}:")
    print(f"  Accuracy: {values['accuracy']}")
    #print(f"  Precision: {values['precision']}")
    #print(f"  Recall: {values['recall']}")
    print(f"  F1 Score: {values['f1_score']}")
    #print(f"  Confusion Matrix:\n{values['confusion_matrix']}\n")

Metrics for AND:
  Accuracy: 0.124
  F1 Score: 0.032
Metrics for OR:
  Accuracy: 0.281
  F1 Score: 0.053
Metrics for NOT:
  Accuracy: 0.727
  F1 Score: 0.129
Metrics for DEPTH:
  Accuracy: 0.2
  F1 Score: 0.079
